# Module 10  Part1
BAM3034-- Hitham Jleed

In [1]:

import warnings
warnings.filterwarnings("ignore")

from pprint import pprint


In [12]:
#pip install --upgrade ipywidgets notebook jupyterlab

## Text Classification & Topic Modeling
*Text classification* is a supervised machine learning problem, where a text document or article classified into a pre-defined set of classes. Topic modeling is the process of discovering groups of co-occurring words in text documents. These group co-occurring related words makes "topics". It is a form of unsupervised learning, so the set of possible topics are unknown. Topic modeling can be used to solve the text classification problem. Topic modeling will identify the topics presents in a document" while text classification classifies the text into a single class.

*Topic Modeling* automatically discover the hidden themes from given documents. It is an unsupervised text analytics algorithm that is used for finding the group of words from the given document. These group of words represents a topic. There is a possibility that, a single document can associate with multiple themes. for example, a group words such as 'patient', 'doctor', 'disease', 'cancer', ad 'health' will represents topic 'healthcare'. Topic Modeling is a different game compared to rule-based text searching that uses regular expressions

![alt text](https://images.datacamp.com/image/upload/f_auto,q_auto:best/v1538411402/image2_ndnai9.png "classification")

# Part A — Document Summarization

Document summarization creates a shorter version of a longer text while preserving its key ideas.

### Two main types
- **Extractive summarization**: selects important sentences directly from the original text.
- **Abstractive summarization**: generates new sentences that paraphrase the original content.

According to the module PDF, summarization is commonly used in:
- news aggregation,
- search engines,
- research article reviews.  



| Feature | Extractive | Abstractive |
| :--- | :--- | :--- |
| **Source Material** | Uses existing phrases | Generates new phrases |
| **Complexity** | Low to Medium | High |
| **Grammar** | Always matches source | Can be more fluid/natural |
| **Risk** | Can be disjointed | Can be factually inaccurate |


## implementation of Extractive Summarization
Best method?

→ LSA (Latent Semantic Analysis) usually gives the most coherent and meaningful summary

→ LexRank (News & Multi-document) is great for news/articles

→ TextRank (General web text) is the original PageRank-inspired method

→ luhn (Quick, dirty technical text) It's fast but often misses the nuance

In [3]:
# Run this single cell
# Uncomment if you need to install
#!pip install sumy nltk

In [4]:
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download('punkt')

# Import everything from sumy first
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words

# Different summarizers you can try
from sumy.summarizers.lsa import LsaSummarizer          # Best overall for coherence
from sumy.summarizers.lex_rank import LexRankSummarizer # Graph-based, very popular
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.luhn import LuhnSummarizer
from sumy.summarizers.edmundson import EdmundsonSummarizer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
# Sample long text (replace with any article, review, or document)
text = """
Artificial intelligence is transforming every industry at an unprecedented pace.
Companies that adopt AI early are gaining massive competitive advantages in efficiency,
customer experience, and innovation. From healthcare diagnostics to autonomous vehicles,
AI models are now outperforming humans in many specialized tasks. However, concerns about
bias, ethics, and job displacement continue to grow. Governments around the world are
rushing to create regulatory frameworks while tech giants invest billions in research.
The future of work, education, and society itself will be shaped by how responsibly we
develop and deploy these powerful technologies. Experts predict that by 2030, AI will
contribute over $15 trillion to the global economy, but only if we solve the current
challenges around transparency and fairness.
"""

In [6]:
import nltk
nltk.download('punkt_tab') # Explicitly download punkt_tab if general punkt isn't sufficient

# CELL 4: Function to perform extractive summarization
def extractive_summary(text, sentence_count=3, method='lsa', language="english"):
    # Parse the text
    parser = PlaintextParser.from_string(text.strip(), Tokenizer(language))

    # Choose summarizer
    if method == 'lsa':
        summarizer = LsaSummarizer(Stemmer(language))
    elif method == 'lexrank':
        summarizer = LexRankSummarizer(Stemmer(language))
    elif method == 'textrank':
        summarizer = TextRankSummarizer(Stemmer(language))
    elif method == 'luhn':
        summarizer = LuhnSummarizer(Stemmer(language))
    else:
        summarizer = LsaSummarizer(Stemmer(language))

    # Set stop words
    summarizer.stop_words = get_stop_words(language)

    # Generate summary
    summary = summarizer(parser.document, sentence_count)

    print(f"Method: {method.upper()} | Sentences: {sentence_count}\n")
    print("SUMMARY:")
    for i, sentence in enumerate(summary, 1):
        print(f"{i}. {sentence}")
    print("\n" + "—"*80)

# Test all methods
extractive_summary(text, sentence_count=3, method='lsa')
extractive_summary(text, sentence_count=3, method='lexrank')
extractive_summary(text, sentence_count=3, method='textrank')
extractive_summary(text, sentence_count=3, method='luhn')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Method: LSA | Sentences: 3

SUMMARY:
1. Companies that adopt AI early are gaining massive competitive advantages in efficiency, customer experience, and innovation.
2. Governments around the world are rushing to create regulatory frameworks while tech giants invest billions in research.
3. Experts predict that by 2030, AI will contribute over $15 trillion to the global economy, but only if we solve the current challenges around transparency and fairness.

————————————————————————————————————————————————————————————————————————————————
Method: LEXRANK | Sentences: 3

SUMMARY:
1. Artificial intelligence is transforming every industry at an unprecedented pace.
2. Companies that adopt AI early are gaining massive competitive advantages in efficiency, customer experience, and innovation.
3. From healthcare diagnostics to autonomous vehicles, AI models are now outperforming humans in many specialized tasks.

————————————————————————————————————————————————————————————————————————————————
Met

# Implement Abstractive Summarization


In [7]:
#!pip install transformers==4.37.2 accelerate sentencepiece

In [8]:
import transformers
print(transformers.__version__)

4.37.2


In [9]:
from transformers import pipeline
import torch

# Load BART-large-cnn — the best general-purpose abstractive model
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    tokenizer="facebook/bart-large-cnn",
    framework="pt",
    device=0 if torch.cuda.is_available() else -1   # Uses GPU if available
)

print("BART-large-cnn summarizer loaded successfully!\n")

BART-large-cnn summarizer loaded successfully!



In [10]:
document = """
Artificial intelligence is transforming every industry at an unprecedented pace.
Tech giants like Google, Microsoft, Amazon, and OpenAI are pouring billions into AI research,
pushing the boundaries of what machines can do. Large language models can now write essays,
code, translate languages, and even pass medical and legal exams. In healthcare, AI is helping
doctors detect cancer earlier and design personalized treatments. Autonomous vehicles are getting
closer to widespread adoption, and robots are entering warehouses and homes. However, these advances
come with serious risks: mass job displacement, deepfakes, algorithmic bias, loss of privacy,
and the potential for autonomous weapons. Experts warn that without strong governance and ethical
guidelines, AI could increase inequality and destabilize society. Many believe the next 5–10 years
will be decisive in determining whether AI becomes the greatest force for human progress or one of
the biggest threats we have ever created.
"""

In [11]:
print("Generating abstractive summary...\n")
summary = summarizer(
    document,
    max_length=150,
    min_length=50,
    do_sample=False,           # False = greedy decoding (more consistent)
    truncation=True            # Auto-truncate if text is too long
)

# Step 5: Display the beautiful result
print("ABSTRACTIVE SUMMARY (BART-large-cnn):")
print("="*70)
print(summary[0]['summary_text'])
print("="*70)

Generating abstractive summary...

ABSTRACTIVE SUMMARY (BART-large-cnn):
Tech giants like Google, Microsoft, Amazon, and OpenAI are pouring billions into AI research. Large language models can now write essays, code, translate languages, and even pass medical and legal exams. Experts warn that without strong governance and ethical guidelines, AI could increase inequality and destabilize society.
